!pip install numpy==1.23.5 scikit-learn==1.2.2

In [1]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

Saving model.pkl to model.pkl
model.pkl uploaded successfully


In [2]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

Saving vectorizer.pkl to vectorizer.pkl
vectorizer.pkl uploaded successfully


In [4]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

Saving balanced_edufeed_dataset.xlsx to balanced_edufeed_dataset.xlsx
balanced_edufeed_dataset.xlsx uploaded successfully


In [26]:
from google.colab import files
uploaded = files.upload()

for name in uploaded.keys():
    print(f"{name} uploaded successfully")

Saving edufeed_zenml_groq.ipynb to edufeed_zenml_groq.ipynb
edufeed_zenml_groq.ipynb uploaded successfully


In [27]:
import joblib

model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

print("All files loaded successfully ")

All files loaded successfully 


In [28]:
print("Pipeline Stages:")
stages = ["Ingest", "Anonymise", "Embed", "FAISS", "Groq Insights", "Export"]

for i, stage in enumerate(stages, 1):
    print(f"Step {i}: {stage}")

Pipeline Stages:
Step 1: Ingest
Step 2: Anonymise
Step 3: Embed
Step 4: FAISS
Step 5: Groq Insights
Step 6: Export


In [29]:
import pandas as pd

df = pd.read_excel("balanced_edufeed_dataset.xlsx")

print("Total rows:", len(df))
df.head()

Total rows: 35421


,professor_name,school_name,department_name,local_name,state_name,year_since_first_review,star_rating,take_again,diff_index,tag_professor,...,post_month,date_missing_flag,grade_group,help_ratio,difficulty_category,experience_bucket,comment_length_cat,tag_count,course_mode,sentiment_label
0,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,6.0,0,B Range,0.0,Moderate,10+ yrs,Medium,3,Offline,Positive
1,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,4.0,0,A Range,0.0,Easy,10+ yrs,Medium,3,Offline,Positive
2,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,7.0,0,Not Reported,0.0,Moderate,10+ yrs,Medium,3,Offline,Positive
3,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,8.0,0,A Range,0.0,Moderate,10+ yrs,Long,3,Offline,Positive
4,Leslie Looney,University Of Illinois At Urbana-Champaign,Astronomy Department,Champaign\Xe2\X80\X93Urbana,Il,11,4.7,0.75,2.0,Hilarious (2) GROUP PROJECTS (2) Gives good ...,...,2.0,0,Not Reported,0.0,Easy,10+ yrs,Long,3,Offline,Positive


In [30]:
print(df.isnull().sum())

professor_name        0
school_name           0
department_name       0
local_name            0
state_name            0
                     ..
experience_bucket     0
comment_length_cat    0
tag_count             0
course_mode           0
sentiment_label       0
Length: 62, dtype: int64


In [31]:
print(df.columns)

Index(['professor_name', 'school_name', 'department_name', 'local_name',
       'state_name', 'year_since_first_review', 'star_rating', 'take_again',
       'diff_index', 'tag_professor', 'num_student', 'post_date',
       'name_onlines', 'name_not_onlines', 'student_star', 'student_difficult',
       'attence', 'for_credits', 'would_take_agains', 'grades', 'help_useful',
       'help_not_useful', 'comments', 'word_comment', 'gender', 'race',
       'asian', 'hispanic', 'nh_black', 'nh_white', 'gives_good_feedback',
       'caring', 'respected', 'participation_matters',
       'clear_grading_criteria', 'skip_class', 'amazing_lectures',
       'inspirational', 'tough_grader', 'hilarious', 'get_ready_to_read',
       'lots_of_homework', 'accessible_outside_class', 'lecture_heavy',
       'extra_credit', 'graded_by_few_things', 'group_projects', 'test_heavy',
       'so_many_papers', 'beware_of_pop_quizzes', 'IsCourseOnline',
       'post_year', 'post_month', 'date_missing_flag', 'grade_g

In [32]:
import re
import pandas as pd

def contains_sensitive(text):
    if pd.isna(text):
        return False

    email = re.search(r'\S+@\S+', str(text))
    numbers = re.search(r'\d{10}', str(text))

    return bool(email or numbers)

In [33]:
df["sensitive_flag"] = df["comments"].apply(contains_sensitive)

In [34]:
col = df.columns[0]  # or manually set correct name

df["sensitive_flag"] = df[col].apply(contains_sensitive)

print("Sensitive data found:", df["sensitive_flag"].sum())

Sensitive data found: 0


In [35]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu

In [36]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

In [17]:
from sentence_transformers import SentenceTransformer
model_embed = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
texts = df["comments"].astype(str).tolist()

In [38]:
import time

start = time.time()

embeddings = model_embed.encode(texts[:500])  # use sample first

end = time.time()

print("Embedding time:", end - start)

Embedding time: 14.958302021026611


In [39]:
!pip install -q faiss-cpu

In [40]:
import faiss
import numpy as np

dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index created with", index.ntotal, "vectors")

FAISS index created with 500 vectors


In [41]:
query = embeddings[0].reshape(1, -1)

start = time.time()
D, I = index.search(query, k=5)
end = time.time()

print("Search time:", end - start)

Search time: 0.0007884502410888672


In [42]:
import json

results = {
    "sample_results": I.tolist(),
    "distances": D.tolist()
}

with open("results.json", "w") as f:
    json.dump(results, f)

print("Results exported")

Results exported


In [44]:
!ls

 balanced_edufeed_dataset.xlsx	 results.json		       vectorizer.pkl
 edufeed_zenml_groq.ipynb	 sample_data
 model.pkl			'tensorflow-model (3).ipynb'


In [45]:
import json

with open("results.json", "r") as f:
    data = json.load(f)

print("Export verified ")

Export verified 
